# v3 - 1. Prepare the data
Public data (cleaned, exact + near duplicates removed) + the hard set written for this project -> families of near-identical texts -> 10 buckets (test = 2, validation = 1, train = 7) -> mixed pairs with the TOAD label rule, built **separately** for the `public` and `hard` regimes.

In [1]:
import os, sys
from pathlib import Path
os.environ["TOADWATCH_VERSION"] = "v3"          # selects the strict "true accuracy" study (v3); keep this line first
sys.path.insert(0, str(Path.cwd().parent))       # notebooks live in ToadWatch_PR/notebooks
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 190); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
from toadwatch import config as C, workflow_v3 as W, protocol_v3 as P, charts_v3 as CH, reporting as R
print("version:", C.VERSION, "| seed:", C.SEED, "| outputs:", C.OUTPUTS_DIR)

version: v3 | seed: 2028 | outputs: /home/rush/Downloads/Fase/ToadWatch_PR/outputs/v3


In [2]:
# After the final test the data is frozen: the saved version is loaded instead of rebuilt.
out = W.load_prepared() if C.FINAL_TEST_LOCK.exists() else W.prepare()      # ~3 minutes
rec = out['records']
print(len(rec), 'records')
display(rec.groupby(['regime', 'split']).size().unstack())
display(rec.groupby(['split', 'pair_type']).size().unstack())

4318 records


split,test,train,val
regime,,,
hard,145,406,51
public,744,2601,371


pair_type,FF,FS,SF,SS
split,,,,
test,355,134,134,266
train,1200,454,454,899
val,168,64,64,126


## The label rule
`FF` = fraud call + fraud message = **fraud (label 1)**; `FS`, `SF`, `SS` = **not fraud (label 0)**.

In [3]:
display(rec.groupby('pair_type').agg(label=('label', 'first'), records=('label', 'size')))
rec[rec.regime == 'hard'].groupby('pair_type').head(1)[['pair_type', 'label', 'call_text', 'msg_text']]

,label,records
pair_type,,
FF,1,1723
FS,0,652
SF,0,652
SS,0,1291


,pair_type,label,call_text,msg_text
4,FF,1,"Hi, Michael here from the federal benefits verification department. There is a pending adjustment on your account th...",Delivery notice: your package could not be delivered because the address is incomplete. Reschedule delivery at.
17,SS,0,"Hola, soy Pedro de su farmacia de la comunidad. Su receta está lista para recoger y puede oprimir 1 para confirmar o...",Aviso de servicio: habrá mantenimiento programado mañana a las 9:30 AM en Aguadilla. Guarde este mensaje como compro...
41,FS,0,"Hola abuela, soy yo, ¿me reconoces? Se me perdió el celular y estoy usando el de un amigo, estoy en un apuro. Solo t...",LUMA Energy: su factura de $18.75 ya está disponible. Guarde este mensaje como comprobante.
80,SF,0,"Hola, soy Carmen de su farmacia de la comunidad. Le llamamos para confirmar su resurtido y para verificar la fecha d...",Parcel update: your package could not be delivered because the address is incomplete. Reschedule delivery at.


## How the families look

In [4]:
import json; print(json.dumps(out['reports']['call_groups'], indent=1)); print(json.dumps(out['reports']['message_groups'], indent=1, default=str))

{
 "texts": 9409,
 "groups": 8592,
 "largest_group": 96,
 "largest_group_share": 0.010202997130407057,
 "texts_in_multi_member_groups": 995,
 "cross_label_near_duplicates_dropped": 0
}
{
 "email": {
  "texts": 13347,
  "groups": 9328,
  "largest_group": 152,
  "largest_group_share": 0.011388326964861018,
  "texts_in_multi_member_groups": 5036,
  "cross_label_near_duplicates_dropped": 6
 },
 "sms": {
  "texts": 10188,
  "groups": 8736,
  "largest_group": 14,
  "largest_group_share": 0.0013741656851197488,
  "texts_in_multi_member_groups": 2416,
  "cross_label_near_duplicates_dropped": 0
 },
 "url": {
  "texts": 73360,
  "groups": 61870
 }
}


## Leak checks on the finished split (all must be True)

In [5]:
from toadwatch.splitting import verify_test_untouched
rec_all = pd.read_parquet(C.PROCESSED_DIR / 'records_all.parquet')
display(P.overlap_checks(rec_all))
print('test fingerprint intact:', verify_test_untouched())

,check,shared,ok
0,call ids shared test/dev,0,True
1,message ids shared test/dev,0,True
2,call families shared test/dev,0,True
3,message families shared test/dev,0,True
4,call families shared fold 0/1,0,True
5,message families shared fold 0/1,0,True
6,call families shared fold 0/2,0,True
7,message families shared fold 0/2,0,True
8,call families shared fold 1/3,0,True
9,message families shared fold 1/3,0,True


test fingerprint intact: True
